Tune SGD regressor

In [1]:
import pandas as pd  
# data = pd.read_csv("data/2025-03-13_data_production.csv",index_col=0)
data = pd.read_csv("data/new_data.csv",index_col=0)
y = pd.read_csv("data/y.csv",index_col=0)['QC_Conc_XY'].astype(float)

In [2]:
# Handeling duplicates
duplicates = data.duplicated(subset=['Batch_XY_name','Batch_XY_date'])
data = data.loc[~duplicates]

# Handeling null values
null_values = list()
for col in data.columns:
    if data[col].isnull().sum() > data.shape[0]/3:
        null_values.append(col)
data.drop(columns=null_values,inplace=True)

# Handeling unique values
variable_to_remove = list()
count_unique = data.astype(str).describe().T

# find columns names that have unique values or identical values (not usefull for ML model)
for i in range(count_unique.shape[0]):
    if count_unique.iloc[i,1] == 280 or count_unique.iloc[i,1] == 1:
        variable_to_remove.append(data.columns[i])
data[variable_to_remove].astype(str).describe()
data.drop(columns=variable_to_remove,inplace=True)


# Data Engineering
data['conc_XX'] = data['Batch_XY_XX_masse'] / data['Batch_XY_YY_Volume']
data.drop(columns=['Batch_XY_XX_masse','Batch_XY_YY_Volume'],inplace=True)


# Handeling time data (transform to datetime format)
datetime_to_remove = list()
for col in data.columns:
    if 'heure' in col or 'date' in col:
        data[col] = pd.to_datetime(data[col])
        datetime_to_remove.append(col)

data['month_production'] = data['Batch_XY_date'].dt.month
data['day_production_start'] = data['Batch_XY_date'].dt.day
data['days_exfo'] = (data['Batch_XY_heure_fin'] - data['Batch_XY_heure_debut']).dt.days

# affiner les batch XX
data.loc[:,'Batch_XY_XX_batch'] = data.Batch_XY_date.dt.year.astype(str) + "-" + data.Batch_XY_XX_batch


In [3]:
# Ajoute data autre source
import joblib
import json
import requests

def get_maree_data(years : list):
    """Cette fonction se connecte à une API externe pour récuperer des données si elle ne sont pas déja présentes"""
    try :
        gde_marees = joblib.load("data/gde_maree.bin")
        print("Donées grandes marées disponibles")
        return gde_marees

    except:
        print("Téléchargement données grandes marées")
        gde_marees = pd.DataFrame()
        for year in years:
            url = f"https://data.stmalo-agglomeration.fr/api/explore/v2.1/catalog/datasets/grandes-marees-a-saint-malo/records?limit=20&refine=date%3A%22{year}%22"
            gde_marees = pd.concat([gde_marees,pd.DataFrame(json.loads(requests.get(url).content)['results'])],axis=0)
        # gde_marees
        gde_marees["date"] = pd.to_datetime(gde_marees["date"])
        joblib.dump(gde_marees,"data/gde_maree.bin")
        return gde_marees
    
def check_if_within_range(row, check_times):
    """
    Compte le nombre de datetimes dans check_times qui tombent dans la plage
    définie par Heure_debut et Heure_ajout2.
    """
    return sum(row["Batch_XY_heure_debut"] <= check_time <= row["Batch_XY_heure_fin"] for check_time in check_times)


gde_marees = get_maree_data(years = [2022,2023,2024,2025])
data["exfo_gde_maree"] =  data.apply(lambda row: check_if_within_range(row, gde_marees['date']), axis=1)
data.drop(columns=datetime_to_remove, inplace=True)


Donées grandes marées disponibles


In [14]:
# data.select_dtypes('number').plot()
# data.drop(columns=['days_exfo']).select_dtypes('number').plot()
# data.drop(columns=['days_exfo','Batch_XY_Agitation']).select_dtypes('number').plot()
# data[['month_production','exfo_gde_maree']].plot()
# data.select_dtypes('number').boxplot()
# import matplotlib.pyplot as plt
# plt.show()

days exfo ont des valeurs absurdes remplacees par la moyenne

les valeurs negatives mise positives

on remplace les variables environementales nulles par la moyenne des temperature pour un même mois

In [4]:
# impute valeur absurdes
import numpy as np

for var in ['Batch_XY_Temperature','Batch_XY_room_HR','Batch_XY_room_T','conc_XX','days_exfo','exfo_gde_maree']:
    _ = data.loc[data[var] != 0, ["month_production",var]]
    _dict = _.groupby(['month_production'])[var].mean().to_dict()

    if not var in ['days_exfo','exfo_gde_maree']:
        data.loc[data[var] == 0, var] = data.loc[data[var] == 0, 'month_production'].map(_dict)
    else:
        Q1 = np.percentile(data[var], 25)
        Q3 = np.percentile(data[var], 75)
        IQR = Q3 - Q1
        # Définir les seuils pour les outliers
        low_fly = Q1 - 1.5 * IQR
        up_fly = Q3 + 1.5 * IQR
        data.loc[(data[var] < low_fly) | (data[var]> up_fly),var] = data.loc[(data[var] < low_fly) | (data[var]> up_fly), 'month_production'].map(_dict)


/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_29216/3561065607.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[103.21621622  -1.75         3.35294118   3.35294118 370.54545455
   5.64        22.9047619   22.9047619  103.21621622   4.23076923
 -39.6875     -39.6875     -39.6875     -39.6875    ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[(data[var] < low_fly) | (data[var]> up_fly),var] = data.loc[(data[var] < low_fly) | (data[var]> up_fly), 'month_production'].map(_dict)
/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_29216/3561065607.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 2.66666667  2.66666667  2.66666667  4.5         4.5         4.5
  6.          6.          6.          1.66666667  1.66666667  1.
  1.57142857  1.57142857

In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

categorical_columns = ['Batch_XY_Technicien','Batch_XY_XX_batch', 'Batch_XY_Analyses',
                        'month_production','day_production_start','exfo_gde_maree']

numerical_columns = ['Batch_XY_Temperature','Batch_XY_Agitation','Batch_XY_room_HR',
                    'conc_XX','Batch_XY_room_T','days_exfo']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(),numerical_columns),
        ('categorical',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),categorical_columns)])

preprocessor.fit(data)

ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                 ['Batch_OGD_Temperature',
                                  'Batch_OGD_Agitation', 'Batch_OGD_room_HR',
                                  'conc_KC8', 'Batch_OGD_room_T',
                                  'days_exfo']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Batch_OGD_Technicien', 'Batch_OGD_KC8_batch',
                                  'Batch_OGD_Analyses', 'month_production',
                                  'day_production_start', 'exfo_gde_maree'])])

In [6]:
X=preprocessor.fit_transform(data)



In [80]:
X.shape

(280, 134)

In [81]:
y.shape

(280,)

# elastic net

In [7]:
from sklearn.linear_model import ElasticNet

best_alpha,best_l1_ratio = 0.15,1
model = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, fit_intercept=True, 
                    precompute=True, max_iter=10000, tol=0.0001,
                    copy_X=True, positive=True, random_state=42, selection='cyclic')

In [14]:
X

array([[-0.64300526,  0.01332488, -0.57933508, ...,  0.        ,
         0.        ,  0.        ],
       [-0.64300526,  0.01332488,  0.17951492, ...,  0.        ,
         0.        ,  0.        ],
       [-0.64300526,  0.01332488, -0.2719528 , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.38727355,  0.01332488,  1.31298707, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.38727355,  0.01332488,  1.31298707, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.38727355,  0.01332488,  1.31298707, ...,  0.        ,
         0.        ,  0.        ]])

In [15]:
col_names=[a.replace('__','_') for a in preprocessor.get_feature_names_out()]
model.fit(pd.DataFrame(X,columns=col_names),y)

ElasticNet(alpha=0.15, l1_ratio=1, max_iter=10000, positive=True,
           precompute=True, random_state=42)

In [ ]:
({a:b for a,b in zip(model.feature_names_in_[model.coef_>0],model.coef_[model.coef_>0])})

{'numerical_Batch_OGD_Temperature': np.float64(0.012477804997146187),
 'numerical_conc_KC8': np.float64(0.7078116278238826),
 'numerical_Batch_OGD_room_T': np.float64(0.005770223507271816),
 'categorical_Batch_OGD_Analyses_0': np.float64(1.4430765838690858)}

In [ ]:
({a.replace('__','_'):b for a,b in zip(preprocessor.get_feature_names_out()[model.coef_>0],model.coef_[model.coef_>0])})


{'numerical_Batch_OGD_Temperature': np.float64(0.012477804997146187),
 'numerical_conc_KC8': np.float64(0.7078116278238826),
 'numerical_Batch_OGD_room_T': np.float64(0.005770223507271816),
 'categorical_Batch_OGD_Analyses_0': np.float64(1.4430765838690858)}

In [12]:
model.feature_names_in_

AttributeError: 'ElasticNet' object has no attribute 'feature_names_in_'

In [65]:
preprocessor.get_feature_names_out()

array(['numerical__Batch_OGD_Temperature',
       'numerical__Batch_OGD_Agitation', 'numerical__Batch_OGD_room_HR',
       'numerical__conc_KC8', 'numerical__Batch_OGD_room_T',
       'numerical__days_exfo', 'categorical__Batch_OGD_Technicien_CD',
       'categorical__Batch_OGD_Technicien_FB',
       'categorical__Batch_OGD_Technicien_IT',
       'categorical__Batch_OGD_Technicien_JP',
       'categorical__Batch_OGD_Technicien_MM',
       'categorical__Batch_OGD_Technicien_NM',
       'categorical__Batch_OGD_Technicien_RS',
       'categorical__Batch_OGD_KC8_batch_2023-K MIX',
       'categorical__Batch_OGD_KC8_batch_2023-K01',
       'categorical__Batch_OGD_KC8_batch_2023-K02',
       'categorical__Batch_OGD_KC8_batch_2023-K03',
       'categorical__Batch_OGD_KC8_batch_2023-K04',
       'categorical__Batch_OGD_KC8_batch_2023-K05',
       'categorical__Batch_OGD_KC8_batch_2023-K07',
       'categorical__Batch_OGD_KC8_batch_2023-K10',
       'categorical__Batch_OGD_KC8_batch_2023-K12',


In [62]:
model.coef_

array([ 0.91476183, 37.84240157, 76.40170959,  0.5257339 ,  0.        ,
       83.36591272, 89.16293975, 15.99097136,  0.        ,  0.        ,
       12.33468398,  0.        , 13.98431027,  0.        ,  0.        ,
       28.68640176,  0.        ,  0.        ,  0.        ,  0.        ])

# sgd regr


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression

# Créer une instance de SGDRegressor
sgd_regressor = SGDRegressor(random_state=42)

# Définir l'espace des hyperparamètres
param_dist = {
    'loss': ['squared_error', 'huber', 'epsilon_insensitive'],
    'penalty': ['l2', 'l1', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
    'eta0': [0.01, 0.1, 0.2],
    'max_iter': [500, 1000, 1500]
}

# Effectuer la recherche aléatoire
random_search = RandomizedSearchCV(sgd_regressor, param_distributions=param_dist, 
                                   n_iter=100, cv=5, scoring='neg_mean_squared_error', 
                                   random_state=42, return_train_score=True)
random_search.fit(X, y)

# Meilleurs hyperparamètres
best_params = random_search.best_params_
print("Meilleurs hyperparamètres :", best_params)

# Évaluer le modèle avec les meilleurs hyperparamètres
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Erreur quadratique moyenne (MSE) :", mse)


/Applications/anaconda3/envs/ML_Flow/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/Applications/anaconda3/envs/ML_Flow/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/Applications/anaconda3/envs/ML_Flow/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/Applications/anaconda3/envs/ML_Flow/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter t

Meilleurs hyperparamètres : {'penalty': 'l1', 'max_iter': 1500, 'loss': 'squared_error', 'learning_rate': 'adaptive', 'eta0': 0.2, 'alpha': 0.001}
Erreur quadratique moyenne (MSE) : 0.00983508587656355


In [43]:
pd.DataFrame(random_search.cv_results_).sort_values(by='rank_test_score').loc[9,:]

mean_fit_time                                                   0.035917
std_fit_time                                                    0.001156
mean_score_time                                                 0.000267
std_score_time                                                  0.000075
param_penalty                                                         l1
param_max_iter                                                      1000
param_loss                                           epsilon_insensitive
param_learning_rate                                           invscaling
param_eta0                                                          0.01
param_alpha                                                       0.0001
params                 {'penalty': 'l1', 'max_iter': 1000, 'loss': 'e...
split0_test_score                                              -0.011837
split1_test_score                                              -0.008851
split2_test_score                                  

In [44]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import ShuffleSplit

model = SGDRegressor(**best_params)
results = cross_validate(model,X,y,cv=ShuffleSplit(n_splits=15,test_size=0.2),
                    return_train_score=True,
                    scoring="neg_root_mean_squared_error")


In [57]:
results['fit_time'].shape[0]

15

# mlflow load

In [87]:
import mlflow

mlflow.set_tracking_uri('http://127.0.0.1:8080')

In [97]:
pd.DataFrame(X[:2,:])

,0,1,2,3,4,5,6,7,8,9,...,124,125,126,127,128,129,130,131,132,133
0,-0.643005,0.013325,-0.579335,-1.443526,-0.912813,-0.075689,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.643005,0.013325,0.179515,-1.443526,-0.305137,-0.075689,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
from utils import *

data = pd.read_csv("data/new_data.csv",index_col=0)
print(data.shape)
n_data = clean_data(data)
print(n_data.shape)

(280, 15)
Donées grandes marées disponibles
(280, 12)


/Users/remicazelles/Documents/Travail/2023-SImplon_microsoft/PROJET_FINAL/ML_supervized/utils.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[103.21621622  -1.75         3.35294118   3.35294118 370.54545455
   5.64        22.9047619   22.9047619  103.21621622   4.23076923
 -39.6875     -39.6875     -39.6875     -39.6875    ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[(data[var] < low_fly) | (data[var]> up_fly),var] = data.loc[(data[var] < low_fly) | (data[var]> up_fly), 'month_production'].map(_dict)
/Users/remicazelles/Documents/Travail/2023-SImplon_microsoft/PROJET_FINAL/ML_supervized/utils.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 2.66666667  2.66666667  2.66666667  4.5         4.5         4.5
  6.          6.          6.          1.66666667  1.6

In [100]:
import mlflow
logged_model = 'runs:/a0e690c63bfe4bc0935639a0a26e8dff/best_model_on_{date2}'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame(X[0,:]).T) # ONE SMAPLE
loaded_model.predict(pd.DataFrame(X[:2,:])) # > 1 SMAPLE

array([0.91857562, 1.0699933 ])

In [88]:
model = mlflow.sklearn.load_model(model_uri="runs:/85dd2fb56e1645e186d82755bef00c16/data/model")


OSError: No such file or directory: '/Users/remicazelles/Documents/Travail/2023-SImplon_microsoft/PROJET_FINAL/ML_supervized/artifcat_RC/526953738941331895/85dd2fb56e1645e186d82755bef00c16/artifacts/data/model'